## 1. Installation & Imports

In [1]:
import gc
import itertools
import json
import os
import pickle
import warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.nn.functional as F
from imblearn.under_sampling import RandomUnderSampler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

print("✅ All imports successful!")

✅ All imports successful!


## 2. Device & Configuration

In [2]:
# Device setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# Check available GPUs
if torch.cuda.is_available():
    print(f"Number of GPUs available: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")

# Global configuration
SEED = 42
SAMPLING_RATIO = 0.01  # Undersample to 1% minority class
BATCH_SIZE = 512
EPOCHS = 50
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-5
CONTEXT_SIZE = 64  # Number of nearest neighbors
N_SPLITS = 5
USE_MULTI_GPU = torch.cuda.device_count() >= 2  # Enable multi-GPU if available

# TabR Model hyperparameters
D_MAIN = 128  # Embedding dimension
D_MULTIPLIER = 2.0
ENCODER_N_BLOCKS = 2
PREDICTOR_N_BLOCKS = 2
DROPOUT_RATE = 0.1
CONTEXT_DROPOUT = 0.1

print("\n" + "="*60)
print("CONFIGURATION")
print("="*60)
print(f"Batch Size: {BATCH_SIZE}")
print(f"Learning Rate: {LEARNING_RATE}")
print(f"D_Main: {D_MAIN}")
print(f"Context Size (k-NN): {CONTEXT_SIZE}")
print(f"Sampling Ratio: {SAMPLING_RATIO}")
print(f"Multi-GPU Enabled: {USE_MULTI_GPU}")
if USE_MULTI_GPU:
    print(f"Number of GPUs to use: {torch.cuda.device_count()}")

Device: cuda
Number of GPUs available: 1
  GPU 0: Tesla P100-PCIE-16GB

CONFIGURATION
Batch Size: 512
Learning Rate: 0.0003
D_Main: 128
Context Size (k-NN): 64
Sampling Ratio: 0.01
Multi-GPU Enabled: False


## 3. Paths & Data Configuration

In [3]:
# Paths
root = Path("/kaggle/input/isic-2024-challenge")
train_path = root / "train-metadata.csv"
test_path = root / "test-metadata.csv"
subm_path = root /  "sample_submission.csv"

id_col = "isic_id"
target_col = "target"
group_col = "patient_id"

err = 1e-5

# Feature column definitions (same as original)
num_cols = [
    "age_approx", "clin_size_long_diam_mm",
    "tbp_lv_A", "tbp_lv_Aext", "tbp_lv_B", "tbp_lv_Bext",
    "tbp_lv_C", "tbp_lv_Cext", "tbp_lv_H", "tbp_lv_Hext",
    "tbp_lv_L", "tbp_lv_Lext", "tbp_lv_areaMM2", "tbp_lv_area_perim_ratio",
    "tbp_lv_color_std_mean", "tbp_lv_deltaA", "tbp_lv_deltaB", "tbp_lv_deltaL",
    "tbp_lv_deltaLB", "tbp_lv_deltaLBnorm", "tbp_lv_eccentricity",
    "tbp_lv_minorAxisMM", "tbp_lv_nevi_confidence", "tbp_lv_norm_border",
    "tbp_lv_norm_color", "tbp_lv_perimeterMM", "tbp_lv_radial_color_std_max",
    "tbp_lv_stdL", "tbp_lv_stdLExt", "tbp_lv_symm_2axis", "tbp_lv_symm_2axis_angle",
    "tbp_lv_x", "tbp_lv_y", "tbp_lv_z",
]

new_num_cols = [
    "lesion_size_ratio", "lesion_shape_index", "hue_contrast", "luminance_contrast",
    "lesion_color_difference", "border_complexity", "color_uniformity",
    "position_distance_3d", "perimeter_to_area_ratio", "area_to_perimeter_ratio",
    "lesion_visibility_score", "symmetry_border_consistency", "consistency_symmetry_border",
    "color_consistency", "consistency_color", "size_age_interaction",
    "hue_color_std_interaction", "lesion_severity_index", "shape_complexity_index",
    "color_contrast_index", "log_lesion_area", "normalized_lesion_size",
    "mean_hue_difference", "std_dev_contrast", "color_shape_composite_index",
    "lesion_orientation_3d", "overall_color_difference",
    "symmetry_perimeter_interaction", "comprehensive_lesion_index",
    "color_variance_ratio", "border_color_interaction",
    "border_color_interaction_2", "size_color_contrast_ratio",
    "age_normalized_nevi_confidence", "age_normalized_nevi_confidence_2",
    "color_asymmetry_index", "volume_approximation_3d", "color_range",
    "shape_color_consistency", "border_length_ratio", "age_size_symmetry_index",
    "index_age_size_symmetry",
]

cat_cols = ["sex", "anatom_site_general", "tbp_tile_type", "tbp_lv_location", "tbp_lv_location_simple", "attribution"]
norm_cols = [f"{col}_patient_norm" for col in num_cols + new_num_cols]
all_num_cols = num_cols + new_num_cols + norm_cols + ["count_per_patient"]

print(f"Numerical columns: {len(all_num_cols)}")
print(f"Categorical columns: {len(cat_cols)}")
print(f"Total features: {len(all_num_cols) + len(cat_cols)}")

Numerical columns: 153
Categorical columns: 6
Total features: 159


## 4. Data Loading & Preprocessing

In [4]:
def read_data(path):
    """Load and engineer features using Polars (same as original)"""
    return (
        pl.read_csv(path)
        .with_columns(
            pl.col("age_approx").cast(pl.String).replace("NA", np.nan).cast(pl.Float64),
        )
        .with_columns(
            pl.col(pl.Float64).fill_nan(pl.col(pl.Float64).median()),
        )
        .with_columns(
            lesion_size_ratio              = pl.col("tbp_lv_minorAxisMM") / pl.col("clin_size_long_diam_mm"),
            lesion_shape_index             = pl.col("tbp_lv_areaMM2") / (pl.col("tbp_lv_perimeterMM") ** 2),
            hue_contrast                   = (pl.col("tbp_lv_H") - pl.col("tbp_lv_Hext")).abs(),
            luminance_contrast             = (pl.col("tbp_lv_L") - pl.col("tbp_lv_Lext")).abs(),
            lesion_color_difference        = (pl.col("tbp_lv_deltaA") ** 2 + pl.col("tbp_lv_deltaB") ** 2 + pl.col("tbp_lv_deltaL") ** 2).sqrt(),
            border_complexity              = pl.col("tbp_lv_norm_border") + pl.col("tbp_lv_symm_2axis"),
            color_uniformity               = pl.col("tbp_lv_color_std_mean") / (pl.col("tbp_lv_radial_color_std_max") + err),
        )
        .with_columns(
            position_distance_3d           = (pl.col("tbp_lv_x") ** 2 + pl.col("tbp_lv_y") ** 2 + pl.col("tbp_lv_z") ** 2).sqrt(),
            perimeter_to_area_ratio        = pl.col("tbp_lv_perimeterMM") / pl.col("tbp_lv_areaMM2"),
            area_to_perimeter_ratio        = pl.col("tbp_lv_areaMM2") / pl.col("tbp_lv_perimeterMM"),
            lesion_visibility_score        = pl.col("tbp_lv_deltaLBnorm") + pl.col("tbp_lv_norm_color"),
            symmetry_border_consistency    = pl.col("tbp_lv_symm_2axis") * pl.col("tbp_lv_norm_border"),
            consistency_symmetry_border    = pl.col("tbp_lv_symm_2axis") * pl.col("tbp_lv_norm_border") / (pl.col("tbp_lv_symm_2axis") + pl.col("tbp_lv_norm_border")),
        )
        .with_columns(
            color_consistency              = pl.col("tbp_lv_stdL") / pl.col("tbp_lv_Lext"),
            consistency_color              = pl.col("tbp_lv_stdL") * pl.col("tbp_lv_Lext") / (pl.col("tbp_lv_stdL") + pl.col("tbp_lv_Lext")),
            size_age_interaction           = pl.col("clin_size_long_diam_mm") * pl.col("age_approx"),
            hue_color_std_interaction      = pl.col("tbp_lv_H") * pl.col("tbp_lv_color_std_mean"),
            lesion_severity_index          = (pl.col("tbp_lv_norm_border") + pl.col("tbp_lv_norm_color") + pl.col("tbp_lv_eccentricity")) / 3,
            shape_complexity_index         = pl.col("border_complexity") + pl.col("lesion_shape_index"),
            color_contrast_index           = pl.col("tbp_lv_deltaA") + pl.col("tbp_lv_deltaB") + pl.col("tbp_lv_deltaL") + pl.col("tbp_lv_deltaLBnorm"),
        )
        .with_columns(
            log_lesion_area                = (pl.col("tbp_lv_areaMM2") + 1).log(),
            normalized_lesion_size         = pl.col("clin_size_long_diam_mm") / pl.col("age_approx"),
            mean_hue_difference            = (pl.col("tbp_lv_H") + pl.col("tbp_lv_Hext")) / 2,
            std_dev_contrast               = ((pl.col("tbp_lv_deltaA") ** 2 + pl.col("tbp_lv_deltaB") ** 2 + pl.col("tbp_lv_deltaL") ** 2) / 3).sqrt(),
            color_shape_composite_index    = (pl.col("tbp_lv_color_std_mean") + pl.col("tbp_lv_area_perim_ratio") + pl.col("tbp_lv_symm_2axis")) / 3,
            lesion_orientation_3d          = pl.arctan2(pl.col("tbp_lv_y"), pl.col("tbp_lv_x")),
            overall_color_difference       = (pl.col("tbp_lv_deltaA") + pl.col("tbp_lv_deltaB") + pl.col("tbp_lv_deltaL")) / 3,
        )
        .with_columns(
            symmetry_perimeter_interaction = pl.col("tbp_lv_symm_2axis") * pl.col("tbp_lv_perimeterMM"),
            comprehensive_lesion_index     = (pl.col("tbp_lv_area_perim_ratio") + pl.col("tbp_lv_eccentricity") + pl.col("tbp_lv_norm_color") + pl.col("tbp_lv_symm_2axis")) / 4,
            color_variance_ratio           = pl.col("tbp_lv_color_std_mean") / pl.col("tbp_lv_stdLExt"),
            border_color_interaction       = pl.col("tbp_lv_norm_border") * pl.col("tbp_lv_norm_color"),
            border_color_interaction_2     = pl.col("tbp_lv_norm_border") * pl.col("tbp_lv_norm_color") / (pl.col("tbp_lv_norm_border") + pl.col("tbp_lv_norm_color")),
            size_color_contrast_ratio      = pl.col("clin_size_long_diam_mm") / pl.col("tbp_lv_deltaLBnorm"),
            age_normalized_nevi_confidence = pl.col("tbp_lv_nevi_confidence") / pl.col("age_approx"),
            age_normalized_nevi_confidence_2 = (pl.col("clin_size_long_diam_mm")**2 + pl.col("age_approx")**2).sqrt(),
            color_asymmetry_index          = pl.col("tbp_lv_radial_color_std_max") * pl.col("tbp_lv_symm_2axis"),
        )
        .with_columns(
            volume_approximation_3d        = pl.col("tbp_lv_areaMM2") * (pl.col("tbp_lv_x")**2 + pl.col("tbp_lv_y")**2 + pl.col("tbp_lv_z")**2).sqrt(),
            color_range                    = (pl.col("tbp_lv_L") - pl.col("tbp_lv_Lext")).abs() + (pl.col("tbp_lv_A") - pl.col("tbp_lv_Aext")).abs() + (pl.col("tbp_lv_B") - pl.col("tbp_lv_Bext")).abs(),
            shape_color_consistency        = pl.col("tbp_lv_eccentricity") * pl.col("tbp_lv_color_std_mean"),
            border_length_ratio            = pl.col("tbp_lv_perimeterMM") / (2 * np.pi * (pl.col("tbp_lv_areaMM2") / np.pi).sqrt()),
            age_size_symmetry_index        = pl.col("age_approx") * pl.col("clin_size_long_diam_mm") * pl.col("tbp_lv_symm_2axis"),
            index_age_size_symmetry        = pl.col("age_approx") * pl.col("tbp_lv_areaMM2") * pl.col("tbp_lv_symm_2axis"),
        )
        .with_columns(
            pl.when(pl.col(col).count().over("patient_id") > 1)
            .then((pl.col(col) - pl.col(col).mean().over("patient_id")) / (pl.col(col).std().over("patient_id") + err))
            .otherwise(0.0)
            .alias(f"{col}_patient_norm")
            for col in (num_cols + new_num_cols)
        )
        .with_columns(
            pl.col(col).count().over("patient_id", col).alias(f"{col}_count") for col in cat_cols
        )
        .with_columns(
            count_per_patient = pl.col("isic_id").count().over("patient_id"),
        )
        .to_pandas()
        .set_index(id_col)
    )

def clean_numerical_features(df, cols):
    """Clean NaN and inf values in numerical features"""
    df = df.copy()
    for col in cols:
        if col in df.columns:
            df[col] = df[col].replace([np.inf, -np.inf], np.nan)
            df[col] = df[col].fillna(df[col].median())
    return df

print("✅ Data loading functions defined!")

✅ Data loading functions defined!


## 5. TabR Model Architecture

In [5]:
class ResidualBlock(nn.Module):
    """Residual block with normalization and dropout"""
    def __init__(self, d_main, d_hidden, dropout, activation="relu"):
        super().__init__()
        self.fc1 = nn.Linear(d_main, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_main)
        self.norm1 = nn.LayerNorm(d_main)
        self.norm2 = nn.LayerNorm(d_main)
        self.dropout = nn.Dropout(dropout)
        self.activation = nn.ReLU() if activation == "relu" else nn.GELU()
        
    def forward(self, x):
        residual = x
        x = self.norm1(x)
        x = self.fc1(x)
        x = self.activation(x)
        x = self.dropout(x)
        x = self.fc2(x)
        x = self.dropout(x)
        return x + residual

class TabRModel(nn.Module):
    """TabR: Tabular Deep Learning Meets Nearest Neighbors
    
    Architecture:
    - Encoder: Project features to embedding space
    - Retrieval: KNN search using PyTorch
    - Mixer: Cross-attention over neighbors
    - Predictor: Final MLP head
    """
    def __init__(self, n_features, d_main=D_MAIN, context_size=CONTEXT_SIZE,
                 encoder_n_blocks=ENCODER_N_BLOCKS, predictor_n_blocks=PREDICTOR_N_BLOCKS,
                 dropout=DROPOUT_RATE, context_dropout=CONTEXT_DROPOUT):
        super().__init__()
        
        self.n_features = n_features
        self.d_main = d_main
        self.context_size = context_size
        
        # Encoder: Project input to embedding space
        self.linear = nn.Linear(n_features, d_main)
        self.encoder_blocks = nn.ModuleList([
            ResidualBlock(d_main, int(d_main * D_MULTIPLIER), dropout)
            for _ in range(encoder_n_blocks)
        ])
        
        # Retrieval: Key projection for similarity search
        self.K = nn.Linear(d_main, d_main, bias=False)
        
        # Mixer: Cross-attention over neighbors
        self.label_embedding = nn.Linear(1, d_main)  # Embed labels as values
        self.T_mlp = nn.Sequential(
            nn.Linear(d_main, int(d_main * D_MULTIPLIER)),
            nn.ReLU(),
            nn.Dropout(context_dropout),
            nn.Linear(int(d_main * D_MULTIPLIER), d_main, bias=False)
        )
        self.context_dropout_layer = nn.Dropout(context_dropout)
        
        # Predictor: Final MLP
        self.predictor_blocks = nn.ModuleList([
            ResidualBlock(d_main, int(d_main * D_MULTIPLIER), dropout)
            for _ in range(predictor_n_blocks)
        ])
        self.head = nn.Sequential(
            nn.LayerNorm(d_main),
            nn.Linear(d_main, 1)
        )
        
    def forward(self, x, candidate_x=None, candidate_y=None, is_train=False):
        """Forward pass with retrieval-augmented attention
        
        Args:
            x: Input features (batch_size, n_features)
            candidate_x: Candidate features for retrieval (None = use x)
            candidate_y: Candidate labels (None = not used)
            is_train: Whether in training mode (exclude self from neighbors)
        
        Returns:
            Predictions (batch_size, 1)
        """
        batch_size = x.shape[0]
        
        # 1. ENCODER: Project input to embedding space
        x_embed = self.linear(x)
        for block in self.encoder_blocks:
            x_embed = block(x_embed)
        
        # Compute keys for similarity search
        keys = self.K(x_embed)  # (batch_size, d_main)
        
        # 2. GET CANDIDATES (all training data for retrieval)
        if candidate_x is None:
            candidate_x = x
            candidate_y = torch.zeros(batch_size, 1).to(x.device)  # Dummy labels
        
        # Embed candidates
        candidate_embed = self.linear(candidate_x)
        for block in self.encoder_blocks:
            candidate_embed = block(candidate_embed)
        candidate_keys = self.K(candidate_embed)
        
        # 3. RETRIEVAL: PyTorch KNN search using torch.cdist
        with torch.no_grad():
            # Compute L2 distances between query keys and candidate keys
            # keys: (batch_size, d_main)
            # candidate_keys: (n_candidates, d_main)
            distances = torch.cdist(keys, candidate_keys)  # (batch_size, n_candidates)
            
            # Find k nearest neighbors
            k_search = self.context_size + (1 if is_train else 0)
            k_search = min(k_search, candidate_keys.shape[0])  # Handle case where k > n_candidates
            _, neighbor_idx = torch.topk(distances, k_search, dim=1, largest=False)  # (batch_size, k_search)
            
            # Exclude self if training
            if is_train:
                neighbor_idx = neighbor_idx[:, 1:]  # Remove self from neighbors
        
        # 4. MIXER: Cross-attention over neighbors
        neighbor_keys = candidate_keys[neighbor_idx]  # (batch_size, context_size, d_main)
        
        # Compute attention weights
        with torch.no_grad():
            # Squared L2 distances between keys and neighbor_keys
            key_squared = (keys ** 2).sum(dim=1, keepdim=True)  # (batch_size, 1)
            neighbor_squared = (neighbor_keys ** 2).sum(dim=2, keepdim=True)  # (batch_size, context_size, 1)
            
            # (2 * k @ neighbor_k.T)
            similarity_cross = 2 * torch.bmm(neighbor_keys, keys.unsqueeze(2))  # (batch_size, context_size, 1)
            similarity_cross = similarity_cross.squeeze(2)
            
            # Full L2 distance
            distances = key_squared - similarity_cross + neighbor_squared.squeeze(2)  # (batch_size, context_size)
            distances = torch.clamp(distances, min=0)
        
        # Softmax over neighbors
        attention_weights = F.softmax(-distances, dim=1)  # Negative distance for similarity
        attention_weights = self.context_dropout_layer(attention_weights)
        
        # Mix neighbor information
        if candidate_y is not None:
            neighbor_y = candidate_y[neighbor_idx]  # (batch_size, context_size, 1)
            neighbor_y_embed = self.label_embedding(neighbor_y)  # (batch_size, context_size, d_main)
        else:
            neighbor_y_embed = torch.zeros_like(neighbor_keys)
        
        # Transform key differences
        key_diff = keys.unsqueeze(1) - neighbor_keys  # (batch_size, context_size, d_main)
        key_transform = self.T_mlp(key_diff)  # (batch_size, context_size, d_main)
        
        # Combine: neighbor_y_emb + transform(k - neighbor_k)
        neighbor_values = neighbor_y_embed + key_transform  # (batch_size, context_size, d_main)
        
        # Weighted sum
        context_vec = torch.sum(
            attention_weights.unsqueeze(2) * neighbor_values,
            dim=1
        )  # (batch_size, d_main)
        
        # Residual connection
        x_mixed = x_embed + context_vec
        
        # 5. PREDICTOR: Final MLP
        for block in self.predictor_blocks:
            x_mixed = block(x_mixed)
        
        output = self.head(x_mixed)  # (batch_size, 1)
        
        return output

print("✅ Model classes defined!")

✅ Model classes defined!


## 6. Dataset & Utilities

In [6]:
class TabularDataset(Dataset):
    """PyTorch Dataset for tabular features"""
    def __init__(self, X, y=None):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float() if y is not None else None
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        # Always return tuple for consistent DataLoader behavior
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            # Return dummy label (0.0) for test data
            return self.X[idx], torch.tensor(0.0, dtype=torch.float32)

def compute_pauc(y_true, y_pred, min_tpr=0.80):
    """Compute partial AUC as used in ISIC 2024 competition"""
    v_gt = np.abs(np.asarray(y_true) - 1)
    v_pred = np.array([1.0 - x for x in y_pred])
    max_fpr = abs(1 - min_tpr)
    partial_auc_scaled = roc_auc_score(v_gt, v_pred, max_fpr=max_fpr)
    partial_auc = 0.5 * max_fpr**2 + (max_fpr - 0.5 * max_fpr**2) / (1.0 - 0.5) * (partial_auc_scaled - 0.5)
    return partial_auc

print("✅ Dataset and utilities defined!")

✅ Dataset and utilities defined!


## 7. Main Training Pipeline

In [7]:
print("="*80)
print("TabR: Tabular Deep Learning Meets Nearest Neighbors (ISIC 2024)")
print("="*80)

# Load data
print("\n[1] Loading and preprocessing data...")
df_train = read_data(train_path)
df_test = read_data(test_path)
df_subm = pd.read_csv(subm_path, index_col=id_col)

print(f"Train: {df_train.shape}, Test: {df_test.shape}")
print(f"Positive samples: {df_train[target_col].sum()} ({df_train[target_col].mean()*100:.2f}%)")

# Clean numerical features (handle NaN/inf)
print("[2] Cleaning numerical features...")
df_train = clean_numerical_features(df_train, all_num_cols)
df_test = clean_numerical_features(df_test, all_num_cols)

# Normalize numerical features
print("[3] Normalizing features with StandardScaler...")
scaler = StandardScaler()
df_train[all_num_cols] = scaler.fit_transform(df_train[all_num_cols])
df_test[all_num_cols] = scaler.transform(df_test[all_num_cols])

# Encode categorical features
print("[4] Encoding categorical features...")
encoders = {}
for cat_col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([df_train[cat_col], df_test[cat_col]])
    le.fit(combined)
    df_train[cat_col] = le.transform(df_train[cat_col])
    df_test[cat_col] = le.transform(df_test[cat_col])
    encoders[cat_col] = le

# Prepare features
feature_cols = all_num_cols + cat_cols
X_train = df_train[feature_cols].values.astype(np.float32)
X_test = df_test[feature_cols].values.astype(np.float32)
y_train = df_train[target_col].values.astype(np.float32)
groups = df_train[group_col].values

print(f"Features: {X_train.shape[1]}, Samples: {X_train.shape[0]}")
print("✅ Data preparation complete!")

# Create models directory
os.makedirs('models', exist_ok=True)

# Save preprocessors for later inference
print("\n[Saving preprocessing objects]")
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
print("✅ Scaler saved")

with open('models/encoders.pkl', 'wb') as f:
    pickle.dump(encoders, f)
print("✅ Encoders saved")

# Save feature configuration
feature_config = {
    'num_cols': num_cols,
    'new_num_cols': new_num_cols,
    'cat_cols': cat_cols,
    'norm_cols': norm_cols,
    'all_num_cols': all_num_cols,
    'feature_cols': feature_cols,
    'n_features': X_train.shape[1]
}
with open('models/feature_config.json', 'w') as f:
    json.dump(feature_config, f, indent=2)
print("✅ Feature config saved")

# Save model hyperparameters
model_config = {
    'seed': SEED,
    'batch_size': BATCH_SIZE,
    'epochs': EPOCHS,
    'learning_rate': LEARNING_RATE,
    'weight_decay': WEIGHT_DECAY,
    'context_size': CONTEXT_SIZE,
    'n_splits': N_SPLITS,
    'd_main': D_MAIN,
    'd_multiplier': D_MULTIPLIER,
    'encoder_n_blocks': ENCODER_N_BLOCKS,
    'predictor_n_blocks': PREDICTOR_N_BLOCKS,
    'dropout_rate': DROPOUT_RATE,
    'context_dropout': CONTEXT_DROPOUT
}
with open('models/model_config.json', 'w') as f:
    json.dump(model_config, f, indent=2)
print("✅ Model config saved")

TabR: Tabular Deep Learning Meets Nearest Neighbors (ISIC 2024)

[1] Loading and preprocessing data...
Train: (401059, 179), Test: (3, 168)
Positive samples: 393 (0.10%)
[2] Cleaning numerical features...
[3] Normalizing features with StandardScaler...
[4] Encoding categorical features...
Features: 159, Samples: 401059
✅ Data preparation complete!

[Saving preprocessing objects]
✅ Scaler saved
✅ Encoders saved
✅ Feature config saved
✅ Model config saved


## 8. Cross-Validation Training

In [8]:
print("\n" + "="*80)
print("Starting 5-Fold Stratified Cross-Validation with TabR")
print("="*80)

cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_preds = np.zeros(len(X_train))
test_preds_list = []
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train, y_train, groups=groups)):
    print(f"\n[Fold {fold+1}/{N_SPLITS}]")
    print(f"Train: {len(train_idx)}, Val: {len(val_idx)}")
    
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train[train_idx], y_train[val_idx]
    
    # Undersample training data
    undersampler = RandomUnderSampler(sampling_strategy=SAMPLING_RATIO, random_state=SEED + fold)
    X_tr_resampled, y_tr_resampled = undersampler.fit_resample(X_tr, y_tr)
    print(f"After undersampling: {len(X_tr_resampled)}")
    
    # Create dataloaders with more workers for better GPU utilization
    num_workers = 4 if USE_MULTI_GPU else 0
    train_dataset = TabularDataset(X_tr_resampled, y_tr_resampled)
    val_dataset = TabularDataset(X_val, y_val)
    test_dataset = TabularDataset(X_test)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=num_workers, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    # Model, optimizer, loss
    model = TabRModel(X_train.shape[1]).to(device)
    
    # Apply DataParallel for multi-GPU training
    if USE_MULTI_GPU:
        print(f"Using DataParallel on {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    
    # Training loop
    best_val_pauc = -np.inf
    best_model_state = None
    patience_counter = 0
    
    for epoch in range(EPOCHS):
        # Train
        model.train()
        train_loss = 0
        for batch_idx, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            
            # Prepare candidates (use validation set for diversity)
            X_cand = torch.from_numpy(X_val).float().to(device)
            y_cand = torch.from_numpy(y_val).float().unsqueeze(1).to(device)
            
            optimizer.zero_grad()
            
            # Handle both DataParallel and single GPU cases
            if USE_MULTI_GPU:
                logits = model.module(X_batch, candidate_x=X_cand, candidate_y=y_cand, is_train=True)
            else:
                logits = model(X_batch, candidate_x=X_cand, candidate_y=y_cand, is_train=True)
            
            loss = criterion(logits.squeeze(), y_batch)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            train_loss += loss.item()
        
        # Validate
        model.eval()
        val_preds = []
        with torch.no_grad():
            X_val_t = torch.from_numpy(X_val).float().to(device)
            y_cand = torch.from_numpy(y_val).float().unsqueeze(1).to(device)
            
            for X_batch, _ in val_loader:  # Unpack tuple from DataLoader
                X_batch = X_batch.to(device)
                
                # Handle both DataParallel and single GPU cases
                if USE_MULTI_GPU:
                    logits = model.module(X_batch, candidate_x=X_val_t, candidate_y=y_cand, is_train=False)
                else:
                    logits = model(X_batch, candidate_x=X_val_t, candidate_y=y_cand, is_train=False)
                
                preds = torch.sigmoid(logits).squeeze().cpu().numpy()
                val_preds.extend(preds)
        
        val_preds = np.array(val_preds)
        val_pauc = compute_pauc(y_val, val_preds)
        
        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}: Loss={train_loss/len(train_loader):.4f}, pAUC={val_pauc:.5f}")
        
        # Early stopping
        if val_pauc > best_val_pauc:
            best_val_pauc = val_pauc
            # Handle both DataParallel and single GPU cases
            if USE_MULTI_GPU:
                best_model_state = deepcopy(model.module.state_dict())
            else:
                best_model_state = deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 10:
                print(f"  Early stopping at epoch {epoch+1}")
                break
    
    # Load best model and get OOF predictions
    if USE_MULTI_GPU:
        model.module.load_state_dict(best_model_state)
    else:
        model.load_state_dict(best_model_state)
    model.eval()
    
    val_preds_final = []
    with torch.no_grad():
        X_val_t = torch.from_numpy(X_val).float().to(device)
        y_cand = torch.from_numpy(y_val).float().unsqueeze(1).to(device)
        
        for X_batch, _ in val_loader:  # Unpack tuple from DataLoader
            X_batch = X_batch.to(device)
            
            # Handle both DataParallel and single GPU cases
            if USE_MULTI_GPU:
                logits = model.module(X_batch, candidate_x=X_val_t, candidate_y=y_cand, is_train=False)
            else:
                logits = model(X_batch, candidate_x=X_val_t, candidate_y=y_cand, is_train=False)
            
            preds = torch.sigmoid(logits).squeeze().cpu().numpy()
            val_preds_final.extend(preds)
    
    oof_preds[val_idx] = np.array(val_preds_final)
    fold_pauc = compute_pauc(y_val, oof_preds[val_idx])
    fold_scores.append(fold_pauc)
    print(f"✅ Fold {fold+1} pAUC: {fold_pauc:.5f}")
    
    # Save model weights for this fold (always save module for consistency)
    if USE_MULTI_GPU:
        torch.save(model.module.state_dict(), f'models/model_fold_{fold}.pth')
    else:
        torch.save(model.state_dict(), f'models/model_fold_{fold}.pth')
    
    # Test predictions
    test_preds_fold = []
    with torch.no_grad():
        X_val_t = torch.from_numpy(X_val).float().to(device)
        y_cand = torch.from_numpy(y_val).float().unsqueeze(1).to(device)
        
        for X_batch, _ in test_loader:  # Unpack tuple from DataLoader
            X_batch = X_batch.to(device)
            
            # Handle both DataParallel and single GPU cases
            if USE_MULTI_GPU:
                logits = model.module(X_batch, candidate_x=X_val_t, candidate_y=y_cand, is_train=False)
            else:
                logits = model(X_batch, candidate_x=X_val_t, candidate_y=y_cand, is_train=False)
            
            preds = torch.sigmoid(logits).squeeze().cpu().numpy()
            test_preds_fold.extend(preds)
    
    test_preds_list.append(np.array(test_preds_fold))
    
    # Clean up
    del model, optimizer, criterion
    torch.cuda.empty_cache()
    gc.collect()


Starting 5-Fold Stratified Cross-Validation with TabR

[Fold 1/5]
Train: 329895, Val: 71164
After undersampling: 31310
  Epoch  10: Loss=0.0213, pAUC=0.18328
  Epoch  20: Loss=0.0132, pAUC=0.17987
  Early stopping at epoch 21
✅ Fold 1 pAUC: 0.18647

[Fold 2/5]
Train: 313765, Val: 87294
After undersampling: 31815
  Epoch  10: Loss=0.0224, pAUC=0.17210
  Epoch  20: Loss=0.0140, pAUC=0.17742
  Early stopping at epoch 22
✅ Fold 2 pAUC: 0.18221

[Fold 3/5]
Train: 323414, Val: 77645
After undersampling: 33633
  Epoch  10: Loss=0.0241, pAUC=0.18836
  Epoch  20: Loss=0.0149, pAUC=0.18943
  Early stopping at epoch 24
✅ Fold 3 pAUC: 0.18990

[Fold 4/5]
Train: 317698, Val: 83361
After undersampling: 30704
  Epoch  10: Loss=0.0238, pAUC=0.17860
  Epoch  20: Loss=0.0131, pAUC=0.17744
  Early stopping at epoch 23
✅ Fold 4 pAUC: 0.18088

[Fold 5/5]
Train: 319464, Val: 81595
After undersampling: 31310
  Epoch  10: Loss=0.0229, pAUC=0.17768
  Epoch  20: Loss=0.0139, pAUC=0.17884
  Epoch  30: Loss=0.00

## 9. Results & Submission

In [9]:
print("\n" + "="*80)
print("CROSS-VALIDATION RESULTS")
print("="*80)
print(f"Fold scores: {[f'{s:.5f}' for s in fold_scores]}")
oof_pauc = compute_pauc(y_train, oof_preds)
print(f"Mean pAUC: {np.mean(fold_scores):.5f} ± {np.std(fold_scores):.5f}")
print(f"OOF pAUC: {oof_pauc:.5f}")

# Save all predictions immediately
print("\n" + "="*80)
print("SAVING ALL ARTIFACTS")
print("="*80)

# 1. Save OOF (Out-of-Fold) predictions
print("\n[1] Saving training OOF predictions...")
training_results = pd.DataFrame({
    id_col: df_train.index,
    target_col: y_train,
    'oof_predictions': oof_preds
})
training_results.to_csv('training_results.csv', index=False)
print(f"✅ Saved: training_results.csv ({len(training_results)} samples)")
print(f"   Columns: {list(training_results.columns)}")

# 2. Save test predictions
print("\n[2] Saving test predictions...")
final_test_preds = np.mean(test_preds_list, axis=0)
test_predictions = pd.DataFrame({
    id_col: df_test.index,
    'test_predictions': final_test_preds
})
test_predictions.to_csv('test_predictions.csv', index=False)
print(f"✅ Saved: test_predictions.csv ({len(test_predictions)} samples)")
print(f"   Predictions range: [{final_test_preds.min():.4f}, {final_test_preds.max():.4f}]")
print(f"   Mean: {final_test_preds.mean():.4f}, Std: {final_test_preds.std():.4f}")

# 3. Save submission file directly
print("\n[3] Saving submission file...")
df_subm[target_col] = final_test_preds
df_subm.to_csv('submission.csv')
print(f"✅ Saved: submission.csv ({len(df_subm)} samples)")

# 4. Save cross-validation results summary
print("\n[4] Saving cross-validation summary...")
cv_summary = pd.DataFrame({
    'fold': range(1, N_SPLITS + 1),
    'pAUC_score': fold_scores
})
cv_summary.to_csv('models/cv_summary.csv', index=False)
print(f"✅ Saved: cv_summary.csv")
print(cv_summary)

# 5. Summary of all saved artifacts
print("\n" + "="*80)
print("📦 SAVED ARTIFACTS SUMMARY")
print("="*80)
print("\n🤖 Model Files (in 'models' directory):")
print(f"  • model_fold_0.pth to model_fold_{N_SPLITS-1}.pth - Model weights for each fold")
print(f"  • scaler.pkl - StandardScaler for numerical features")
print(f"  • encoders.pkl - LabelEncoders for categorical features")
print(f"  • feature_config.json - Feature column names and configuration")
print(f"  • model_config.json - Model hyperparameters")
print(f"  • cv_summary.csv - Cross-validation scores")
print("\n📊 Prediction Files:")
print(f"  • training_results.csv - OOF predictions for training data")
print(f"  • test_predictions.csv - Predictions for test data")
print(f"  • submission.csv - Final submission file")
print("\n✅ Ready for inference on new data using 'tabr-isic2024-inference.ipynb'")


CROSS-VALIDATION RESULTS
Fold scores: ['0.18647', '0.18221', '0.18990', '0.18088', '0.18505']
Mean pAUC: 0.18490 ± 0.00319
OOF pAUC: 0.14401

SAVING ALL ARTIFACTS

[1] Saving training OOF predictions...
✅ Saved: training_results.csv (401059 samples)
   Columns: ['isic_id', 'target', 'oof_predictions']

[2] Saving test predictions...
✅ Saved: test_predictions.csv (3 samples)
   Predictions range: [0.0002, 0.0003]
   Mean: 0.0002, Std: 0.0000

[3] Saving submission file...
✅ Saved: submission.csv (3 samples)

[4] Saving cross-validation summary...
✅ Saved: cv_summary.csv
   fold  pAUC_score
0     1    0.186467
1     2    0.182207
2     3    0.189901
3     4    0.180877
4     5    0.185048

📦 SAVED ARTIFACTS SUMMARY

🤖 Model Files (in 'models' directory):
  • model_fold_0.pth to model_fold_4.pth - Model weights for each fold
  • scaler.pkl - StandardScaler for numerical features
  • encoders.pkl - LabelEncoders for categorical features
  • feature_config.json - Feature column names and c

## 10. Create Submission CSV

In [10]:
# Create submission DataFrame
print("\n" + "="*80)
print("CREATING SUBMISSION CSV")
print("="*80)

submission = pd.DataFrame({
    'isic_id': df_test.index,
    'target': final_test_preds
})

print(f"\nSubmission shape: {submission.shape}")
print(f"Target statistics:")
print(f"  Min: {submission['target'].min():.6f}")
print(f"  Max: {submission['target'].max():.6f}")
print(f"  Mean: {submission['target'].mean():.6f}")
print(f"  Median: {submission['target'].median():.6f}")
print(f"  Std: {submission['target'].std():.6f}")

# Save submission
submission.to_csv('submission.csv', index=False)
print(f"\n✅ Submission saved to 'submission.csv'")
print(f"\nFirst 10 rows:")
print(submission.head(10))


CREATING SUBMISSION CSV

Submission shape: (3, 2)
Target statistics:
  Min: 0.000238
  Max: 0.000255
  Mean: 0.000247
  Median: 0.000249
  Std: 0.000009

✅ Submission saved to 'submission.csv'

First 10 rows:
        isic_id    target
0  ISIC_0015657  0.000249
1  ISIC_0015729  0.000255
2  ISIC_0015740  0.000238
